# R3 · Fonctions — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r3_fonctions/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r3_fonctions/03_notebook.ipynb`, créée par `python tools/start_chapter.py R3`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R3.1 à R3.12, R3.15 | Les exercices papier (vérification) | ✏️ 📈 🔗 | | |
| A | R3.21 | Tracer et lire les fonctions de référence | 📦 | ★ | 15 |
|  | R3.26 | Un graphique propre : titre, axes, légende | 🛠️ | ★ | 15 |
| B | R3.18 | La fonction affine qui passe par deux points · `affine_from_points` | 🔨 | ★ | 25 |
|  | R3.19 | Racines et sommet d'un polynôme du second degré · `quadratic_roots`, `quadratic_vertex` | 🔨 | ★★ | 40 |
|  | R3.22 | Faire varier a, b et c : déformer une parabole | 🎨 | ★★ | 20 |
|  | R3.24 | Combien de racines ? Le signe du discriminant sur une grille de paramètres | 🔬 | ★★ | 20 |
| C | R3.20 | Composer deux fonctions · `compose` | 🔨 | ★ | 25 |
|  | R3.23 | Prédire f∘g et g∘f pour « valeur absolue » et « moins 3 » | 🔮 | ★ | 10 |
|  | R3.25 | Chasse à l'erreur : composition, réciproque et racines | 🐛 | ★★ | 20 |
|  | R3.17 | Évaluer un polynôme par Horner · `polyval` | 🔨 | ★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R3")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : images, antécédents, lecture graphique, droites, second degré, variations, composées, réciproques, Horner.
- Écrire les cinq fonctions de `mymath/functions.py` et les tester.
- Tracer des fonctions avec NumPy, déformer des paraboles, prédire des composées.

**Rappel express.** Pente $m = \frac{y_2 - y_1}{x_2 - x_1}$, $c = y_1 - m x_1$ ; $ax^2 + bx + c = a(x - \alpha)^2 + \beta$ avec $\alpha = -\frac{b}{2a}$ ; $\Delta = b^2 - 4ac$ et racines $\frac{-b \pm \sqrt{\Delta}}{2a}$ ; $(f \circ g)(x) = f(g(x))$, $g$ d'abord ; $f^{-1}(f(x)) = x$.

## Partie 0 · Vérifier tes exercices papier (✏️ R3.1 à R3.12, 📈 R3.2, 🔗 R3.15)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un nombre avec un point décimal, `-1.5` ; une fraction exacte entre guillemets, `"5/7"` ;
- une liste de coefficients, du plus haut degré au terme constant, zéros compris : `[2, 0, -5]` pour $2x^2 - 5$ ;
- une liste de valeurs (solutions, antécédents, racines) dans l'ordre que tu veux, la liste vide s'il n'y en a aucune ; une fraction s'y écrit comme un calcul : `[-2/7, 6]` ;
- un intervalle entre guillemets, `"]-1 ; 4]"` ou `"[0 ; +oo["`, une réunion avec un U ;
- une fonction affine par la liste `[m, c]`, une forme canonique par `[a, alpha, beta]` ;
- vrai ou faux par `True` ou `False`.

Les réponses pas encore remplies affichent ⏳.

**Ex R3.1 — Images, antécédents et ensemble de définition**

In [ ]:
# Ex R3.1: the values you found on paper
answer_R3_1a = ...  # a) a number
answer_R3_1b = ...  # b) a fraction, as text
answer_R3_1c = ...  # c) a list
answer_R3_1d = ...  # d) a list (empty if there is none)
answer_R3_1e = ...  # e) a number
answer_R3_1f = ...  # f) a union of intervals, as text
answer_R3_1g = ...  # g) an interval, as text
answer_R3_1h = ...  # h) a union of intervals, as text

for letter, answer in zip("abcdefgh", [answer_R3_1a, answer_R3_1b, answer_R3_1c, answer_R3_1d, answer_R3_1e, answer_R3_1f, answer_R3_1g, answer_R3_1h]):
    wbm.check(f"R3.1{letter}", answer)

**Ex R3.2 — Lire des valeurs, des solutions et des variations sur un graphique**

In [ ]:
# Ex R3.2: the values you found on paper
answer_R3_2a = ...  # a) a number
answer_R3_2b = ...  # b) a number
answer_R3_2c = ...  # c) a list
answer_R3_2d = ...  # d) a list
answer_R3_2e = ...  # e) a number
answer_R3_2f = ...  # f) a list
answer_R3_2g = ...  # g) an interval, as text
answer_R3_2h = ...  # h) a union of intervals, as text

for letter, answer in zip("abcdefgh", [answer_R3_2a, answer_R3_2b, answer_R3_2c, answer_R3_2d, answer_R3_2e, answer_R3_2f, answer_R3_2g, answer_R3_2h]):
    wbm.check(f"R3.2{letter}", answer)

**Ex R3.3 — La fonction affine qui passe par deux points**

In [ ]:
# Ex R3.3: the values you found on paper
answer_R3_3a = ...  # a) the list [m, c]
answer_R3_3b = ...  # b) the list [m, c]
answer_R3_3c = ...  # c) the list [m, c]
answer_R3_3d = ...  # d) a number
answer_R3_3e = ...  # e) a number
answer_R3_3f = ...  # f) True or False
answer_R3_3g = ...  # g) the list [m, c]

for letter, answer in zip("abcdefg", [answer_R3_3a, answer_R3_3b, answer_R3_3c, answer_R3_3d, answer_R3_3e, answer_R3_3f, answer_R3_3g]):
    wbm.check(f"R3.3{letter}", answer)

**Ex R3.4 — Ramener des données entre 0 et 1 avec une fonction affine (min-max)**

In [ ]:
# Ex R3.4: the values you found on paper
answer_R3_4a = ...  # a) the list [m, c]
answer_R3_4b = ...  # b) a number
answer_R3_4c = ...  # c) the list [m, c]
answer_R3_4d = ...  # d) a number (m²)
answer_R3_4e = ...  # e) the list [m, c]

for letter, answer in zip("abcde", [answer_R3_4a, answer_R3_4b, answer_R3_4c, answer_R3_4d, answer_R3_4e]):
    wbm.check(f"R3.4{letter}", answer)

**Ex R3.5 — Variations et signe des fonctions de référence**

In [ ]:
# Ex R3.5: the values you found on paper
answer_R3_5a = ...  # a) an interval, as text
answer_R3_5b = ...  # b) an interval, as text
answer_R3_5c = ...  # c) an interval, as text
answer_R3_5d = ...  # d) an interval, as text
answer_R3_5e = ...  # e) a union of intervals, as text
answer_R3_5f = ...  # f) an interval, as text
answer_R3_5g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_R3_5a, answer_R3_5b, answer_R3_5c, answer_R3_5d, answer_R3_5e, answer_R3_5f, answer_R3_5g]):
    wbm.check(f"R3.5{letter}", answer)

**Ex R3.6 — Forme canonique et sommet d'une parabole**

In [ ]:
# Ex R3.6: the values you found on paper
answer_R3_6a = ...  # a) the list [a, alpha, beta]
answer_R3_6b = ...  # b) the list [a, alpha, beta]
answer_R3_6c = ...  # c) the list [a, alpha, beta]
answer_R3_6d = ...  # d) the list [alpha, beta]
answer_R3_6e = ...  # e) the list [a, b, c]
answer_R3_6f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_R3_6a, answer_R3_6b, answer_R3_6c, answer_R3_6d, answer_R3_6e, answer_R3_6f]):
    wbm.check(f"R3.6{letter}", answer)

**Ex R3.7 — Racines par le discriminant et factorisation**

In [ ]:
# Ex R3.7: the values you found on paper
answer_R3_7a = ...  # a) a number
answer_R3_7b = ...  # b) a list
answer_R3_7c = ...  # c) the list [a, x1, x2]
answer_R3_7d = ...  # d) a list
answer_R3_7e = ...  # e) a list (empty if there is none)
answer_R3_7f = ...  # f) a list
answer_R3_7g = ...  # g) an interval, as text

for letter, answer in zip("abcdefg", [answer_R3_7a, answer_R3_7b, answer_R3_7c, answer_R3_7d, answer_R3_7e, answer_R3_7f, answer_R3_7g]):
    wbm.check(f"R3.7{letter}", answer)

**Ex R3.8 — Trouver le maximum d'une aire ou d'un bénéfice**

In [ ]:
# Ex R3.8: the values you found on paper
answer_R3_8a = ...  # a) a number (m²)
answer_R3_8b = ...  # b) a number (m)
answer_R3_8c = ...  # c) a list
answer_R3_8d = ...  # d) a number (€)
answer_R3_8e = ...  # e) a number (€)

for letter, answer in zip("abcde", [answer_R3_8a, answer_R3_8b, answer_R3_8c, answer_R3_8d, answer_R3_8e]):
    wbm.check(f"R3.8{letter}", answer)

**Ex R3.9 — Composer deux fonctions dans les deux ordres**

In [ ]:
# Ex R3.9: the values you found on paper
answer_R3_9a = ...  # a) a number
answer_R3_9b = ...  # b) a number
answer_R3_9c = ...  # c) a list of coefficients
answer_R3_9d = ...  # d) a list of coefficients
answer_R3_9e = ...  # e) a list of coefficients
answer_R3_9f = ...  # f) the list [m, c]
answer_R3_9g = ...  # g) a list
answer_R3_9h = ...  # h) a list of coefficients

for letter, answer in zip("abcdefgh", [answer_R3_9a, answer_R3_9b, answer_R3_9c, answer_R3_9d, answer_R3_9e, answer_R3_9f, answer_R3_9g, answer_R3_9h]):
    wbm.check(f"R3.9{letter}", answer)

**Ex R3.10 — Trouver une réciproque et son ensemble de définition**

In [ ]:
# Ex R3.10: the values you found on paper
answer_R3_10a = ...  # a) the list [m, c]
answer_R3_10b = ...  # b) a number
answer_R3_10c = ...  # c) a number
answer_R3_10d = ...  # d) a union of intervals, as text
answer_R3_10e = ...  # e) an interval, as text
answer_R3_10f = ...  # f) a number
answer_R3_10g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_R3_10a, answer_R3_10b, answer_R3_10c, answer_R3_10d, answer_R3_10e, answer_R3_10f, answer_R3_10g]):
    wbm.check(f"R3.10{letter}", answer)

**Ex R3.11 — Fonctions par morceaux et argmax**

In [ ]:
# Ex R3.11: the values you found on paper
answer_R3_11a = ...  # a) a number
answer_R3_11b = ...  # b) a number
answer_R3_11c = ...  # c) an interval, as text
answer_R3_11d = ...  # d) a number
answer_R3_11e = ...  # e) an expression in x, as text
answer_R3_11f = ...  # f) a number
answer_R3_11g = ...  # g) a number
answer_R3_11h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R3_11a, answer_R3_11b, answer_R3_11c, answer_R3_11d, answer_R3_11e, answer_R3_11f, answer_R3_11g, answer_R3_11h]):
    wbm.check(f"R3.11{letter}", answer)

**Ex R3.12 — Évaluer un polynôme par le schéma de Horner**

In [ ]:
# Ex R3.12: the values you found on paper
answer_R3_12a = ...  # a) a number
answer_R3_12b = ...  # b) the list of the running values, in order
answer_R3_12c = ...  # c) a number
answer_R3_12d = ...  # d) an integer

for letter, answer in zip("abcd", [answer_R3_12a, answer_R3_12b, answer_R3_12c, answer_R3_12d]):
    wbm.check(f"R3.12{letter}", answer)

**Ex R3.15 — Une fonction de coût donnée par une formule : une parabole et son sommet**

In [ ]:
# Ex R3.15: the values you found on paper
answer_R3_15a = ...  # a) a number
answer_R3_15b = ...  # b) a number
answer_R3_15c = ...  # c) a number
answer_R3_15d = ...  # d) an interval, as text
answer_R3_15e = ...  # e) an interval, as text

for letter, answer in zip("abcde", [answer_R3_15a, answer_R3_15b, answer_R3_15c, answer_R3_15d, answer_R3_15e]):
    wbm.check(f"R3.15{letter}", answer)

## Partie A · Fonctions de référence et graphiques

Fiche §2 et §4. Tracer des fonctions avec NumPy, lire leurs positions relatives, et faire une figure qu'on peut montrer.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R3.21 — Tracer et lire les fonctions de référence 📦 ★ ⏱️ 15 min
**Objectif :** tracer des fonctions avec NumPy et matplotlib, lire leurs positions relatives, puis les justifier par le calcul.  
**Prérequis :** fiche §2 et §4 ; R0 (matplotlib) · **Parcours :** C, E

Avec NumPy, une fonction s'applique à tout un tableau d'un coup : `xs = np.linspace(0, 3, 300)` fabrique 300 abscisses régulières, et `np.sqrt(xs)`, `xs**2`, `np.abs(xs)` ou `1 / xs` calculent les images de toutes ces abscisses.

Complète `plot_reference()` : deux graphiques côte à côte (`fig, axes = plt.subplots(1, 2, figsize=(10, 4))`).
- À gauche, sur $[0 \;;\; 3]$ : les courbes de $x$, de $x^2$ et de $\sqrt{x}$, avec une légende.
- À droite, sur $[-3 \;;\; 3]$ : les courbes de $\frac{1}{x}$ et de $x$. Attention : $\frac{1}{x}$ n'est pas définie en 0 ; trace-la en **deux morceaux**, sur $[-3 \;;\; -0{,}1]$ et sur $[0{,}1 \;;\; 3]$, sinon matplotlib relie les deux branches par un trait vertical qui n'existe pas.

Puis, en lisant tes figures **et** en le justifiant par le calcul dans ta copie :
- a) l'intervalle de $]0 \;;\; +\infty[$ sur lequel $\sqrt{x} > x$ ;
- b) l'ensemble des réels $x$ (pas seulement ceux de la figure) tels que $x^2 > x$ ;
- c) les abscisses des points communs aux courbes de $\frac{1}{x}$ et de $x$.

In [ ]:
def plot_reference():
    """Left: x, x² and √x on [0, 3]. Right: 1/x (in two pieces) and x on [−3, 3]. Legends, titles, axis labels."""
    raise NotImplementedError("trace les deux graphiques")


answer_R3_21a = ...  # a) an interval, as text
answer_R3_21b = ...  # b) an interval or a union, as text
answer_R3_21c = ...  # c) a list

In [ ]:
with checked("R3.21 figure"):
    plot_reference()
for letter, answer in zip("abc", [answer_R3_21a, answer_R3_21b, answer_R3_21c]):
    wbm.check(f"R3.21{letter}", answer)

### Ex R3.26 — Un graphique propre : titre, axes, légende 🛠️ ★ ⏱️ 15 min
**Objectif :** produire une figure qu'on peut montrer : titre, axes nommés avec leur unité, légende, point remarquable annoté.  
**Prérequis :** fiche §2, l'encadré « Un graphique propre » ; R0 (matplotlib) · **Parcours :** C

Une usine estime son coût de production par $C(q) = 0{,}5\,q^2 - 20\,q + 300$, en milliers d'euros, pour $q$ (en tonnes) entre 0 et 40. Son budget est de 150 milliers d'euros.

Complète `plot_cost(ax)`, qui dessine **dans l'axe `ax` qu'on lui donne** :
- la courbe de $C$ sur $[0 \;;\; 40]$ ;
- la droite horizontale du budget (`ax.axhline`) ;
- le point du minimum de $C$ (calcule-le d'abord à la main : §5), marqué d'un rond et annoté (`ax.annotate`) ;
- un titre, les axes nommés **avec leurs unités**, une légende, une grille.

La première cellule de vérification affiche ta figure ; la seconde regarde automatiquement qu'il ne manque rien d'essentiel. Compare ensuite ta figure à celle de `05_solutions.ipynb` : plusieurs mises en forme sont justes. Dans ta copie, lis sur ta figure les productions qui respectent le budget.

In [ ]:
def plot_cost(ax):
    """Draw C(q) = 0.5 q² − 20 q + 300 on [0, 40], the budget line y = 150 and the marked minimum, in ax."""
    raise NotImplementedError("dessine la courbe, le budget et le minimum dans ax")

In [ ]:
with checked("R3.26"):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    try:
        plot_cost(ax)
    except BaseException:
        plt.close(fig)                                 # no empty figure while plot_cost is not written yet
        raise
    plt.show()

**Vérifications automatiques** : ce qu'une figure à montrer doit contenir.

In [ ]:
with checked("R3.26 vérifications"):
    fig, ax = plt.subplots(figsize=(6.5, 4))
    try:
        plot_cost(ax)
    finally:
        plt.close(fig)                                 # only the checks below are shown here
    problems = []
    if not ax.get_title():
        problems.append("un titre (`ax.set_title`)")
    if not ax.get_xlabel() or not ax.get_ylabel():
        problems.append("les deux axes nommés (`ax.set_xlabel`, `ax.set_ylabel`)")
    if ax.get_legend() is None:
        problems.append("une légende (`label=` dans chaque `plot`, puis `ax.legend()`)")
    if len(ax.lines) < 3:
        problems.append("la courbe, la droite du budget et le point du minimum (trois tracés au moins)")
    if problems:
        print("❌ Ex R3.26 : il manque " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R3.26 : titre, axes nommés, légende et trois tracés ; compare ta figure à celle de "
              "05_solutions.ipynb.")

## Partie B · Fonctions affines et second degré

Fiche §3 et §5. Tes premières fonctions de `mymath/functions.py`, une famille de paraboles à déformer, et une grille de discriminants.

### Ex R3.18 — La fonction affine qui passe par deux points · `affine_from_points` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer le calcul de la pente et de l'ordonnée à l'origine, et refuser une droite verticale.  
**Prérequis :** fiche §3 ; workbookIA 0A §100.6.2 (`raise`) · **mymath :** `functions.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en R2). Ouvre ta copie `mon_travail/mymath/functions.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Les tests comparent tes résultats à NumPy et à SymPy : écris le calcul toi-même, sans les appeler.

Écris `affine_from_points(p, q)` : `p` et `q` sont deux points `(x, y)`. Dépaquette-les (`(x1, y1), (x2, y2) = p, q`), lève une `ValueError` si leurs abscisses sont égales (une droite verticale n'est pas la courbe d'une fonction), puis calcule la pente et l'ordonnée à l'origine (fiche §3) ; rends le tuple `(m, c)`, deux flottants.

La vérification calcule `affine_from_points((-1, 4), (3, -4))`, puis le nom de l'erreur levée pour deux points de même abscisse (avec `raised`, de la cellule des outils).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R3")  # reload your saved file

In [ ]:
with checked("R3.18a"):
    wbm.check("R3.18a", list(mymath.functions.affine_from_points((-1, 4), (3, -4))), computed=True)
with checked("R3.18b"):
    wbm.check("R3.18b", raised(mymath.functions.affine_from_points, (2, 1), (2, 5)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r3_functions.py", "test_affine_from_points_")

### Ex R3.19 — Racines et sommet d'un polynôme du second degré · `quadratic_roots`, `quadratic_vertex` 🔨 ★★ ⏱️ 40 min
**Objectif :** programmer les trois cas du discriminant et la forme canonique.  
**Prérequis :** fiche §5 ; R3.18 · **mymath :** `functions.py` · **Parcours :** C, E

Dans `mon_travail/mymath/functions.py` :
- a) `quadratic_roots(a, b, c)` : une `ValueError` si `a == 0` ; sinon le discriminant décide (fiche §5) : deux racines (`math.sqrt`, rendues dans l'ordre croissant, même quand `a` est négatif), une racine double, ou la liste vide. Compare le discriminant à 0 avec `==` : les tests de la racine double utilisent des entiers. Le squelette n'importe pas `math` : ajoute `import math` en haut du fichier.
- b) `quadratic_vertex(a, b, c)` : une `ValueError` si `a == 0` ; sinon le tuple `(alpha, beta)` de la forme canonique.

La vérification calcule `quadratic_roots(2, -7, 3)`, regarde si `quadratic_roots(1, 2, 3)` vaut bien `[]`, puis calcule `quadratic_vertex(-1, 6, -5)`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R3")  # reload your saved file

In [ ]:
with checked("R3.19a"):
    wbm.check("R3.19a", mymath.functions.quadratic_roots(2, -7, 3), computed=True)
with checked("R3.19b"):
    wbm.check("R3.19b", mymath.functions.quadratic_roots(1, 2, 3) == [], computed=True)
with checked("R3.19c"):
    wbm.check("R3.19c", list(mymath.functions.quadratic_vertex(-1, 6, -5)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r3_functions.py", "test_quadratic_roots_ or test_quadratic_vertex_")

### Ex R3.22 — Faire varier a, b et c : déformer une parabole 🎨 ★★ ⏱️ 20 min
**Objectif :** voir ce que chaque paramètre fait à une parabole, et pourquoi la forme canonique le dit d'un coup d'œil.  
**Prérequis :** fiche §5 · **Parcours :** C, E

Complète `plot_families(axes)`, qui reçoit trois axes côte à côte et trace, sur $[-4 \;;\; 4]$ :
1. dans `axes[0]`, la famille $y = a\,x^2$ pour $a \in \{-2, -0{,}5, 0{,}5, 1, 2\}$ ;
2. dans `axes[1]`, la famille $y = x^2 + c$ pour $c \in \{-3, 0, 2\}$ ;
3. dans `axes[2]`, la famille $y = (x - h)^2$ pour $h \in \{-2, 0, 3\}$.

Une légende dans chaque graphique (la valeur du paramètre), un titre, la même fenêtre verticale $[-6 \;;\; 8]$ partout pour comparer.

Puis réponds dans ta copie : que fait $a$ (son signe, sa taille) ? que fait $c$ ? que fait $h$ ? En déduire, sans calcul, à quoi ressemble la courbe de $y = 2(x - 3)^2 - 1$ par rapport à celle de $y = x^2$. Et le coefficient $b$ de $ax^2 + bx + c$ : pourquoi est-il plus difficile à « lire » que $a$, $\alpha$ et $\beta$ ?

In [ ]:
def plot_families(axes):
    """Three families of parabolas: y = a x², y = x² + c, y = (x − h)², one per axis, with legends and titles."""
    raise NotImplementedError("trace les trois familles")

In [ ]:
with checked("R3.22"):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    try:
        plot_families(axes)
    except BaseException:
        plt.close(fig)
        raise
    plt.show()

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R3.22 vérifications"):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    try:
        plot_families(axes)
    finally:
        plt.close(fig)
    problems = []
    if any(len(ax.lines) < 3 for ax in axes):
        problems.append("au moins trois paraboles dans chacun des trois graphiques")
    if not all(ax.get_title() for ax in axes):
        problems.append("un titre par graphique, qui dit quel paramètre varie")
    if problems:
        print("❌ Ex R3.22 : il manque " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R3.22 : trois familles de paraboles, chacune avec son titre ; à toi de conclure dans ta copie.")

### Ex R3.24 — Combien de racines ? Le signe du discriminant sur une grille de paramètres 🔬 ★★ ⏱️ 20 min
**Objectif :** voir sur une grille de paramètres quand un second degré a deux, une ou aucune racine, et trouver la frontière.  
**Prérequis :** fiche §5 ; R3.19 · **Parcours :** C

On étudie tous les polynômes $x^2 + bx + c$ avec $b$ et $c$ **entiers** de $-5$ à $5$ : 121 polynômes.

- a) Écris `count_roots()` : la liste `[n2, n1, n0]`, où `n2` est le nombre de ces polynômes qui ont deux racines, `n1` le nombre qui en ont une, et `n0` le nombre qui n'en ont aucune. Deux boucles `for` imbriquées et le discriminant suffisent (ou ta fonction `quadratic_roots` de R3.19).
- b) Sur la figure de l'expérience, les points « une racine » sont sur une courbe qui sépare les deux autres familles. Écris `boundary(b)`, qui donne le $c$ de cette frontière en fonction de $b$ (il s'agit de résoudre $\Delta = 0$ en $c$), puis donne son expression en `b`, entre guillemets.

L'expérience colore les 121 points selon le signe de $\Delta$ et trace ta frontière par-dessus.

In [ ]:
def count_roots():
    """[n2, n1, n0]: how many x² + bx + c (b, c integers from −5 to 5) have two, one or no real root."""
    raise NotImplementedError("compte avec deux boucles et le discriminant")


def boundary(b):
    """The c for which x² + bx + c has exactly one root (works on a NumPy array of b)."""
    raise NotImplementedError("résous Δ = 0 en c")


answer_R3_24b = ...  # b) the expression of the boundary c in terms of b, as text, e.g. "3*b - 1"

In [ ]:
with checked("R3.24a"):
    wbm.check("R3.24a", count_roots(), computed=True)
wbm.check("R3.24b", answer_R3_24b)

**L'expérience** : les 121 polynômes, colorés selon le signe du discriminant, et ta frontière.

In [ ]:
with checked("R3.24 figure"):
    bs, cs = np.meshgrid(range(-5, 6), range(-5, 6))
    deltas = bs**2 - 4 * cs
    grid = np.linspace(-5, 5, 200)
    frontier = boundary(grid)                     # before the figure: no half-drawn figure if boundary is not written
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for mask, colour, label in [(deltas > 0, "C0", "Δ > 0 : deux racines"), (deltas == 0, "C3", "Δ = 0 : une racine"),
                                (deltas < 0, "C2", "Δ < 0 : aucune racine")]:
        ax.scatter(bs[mask], cs[mask], color=colour, label=label, s=30)
    ax.plot(grid, frontier, color="black", linewidth=1.2, label="ta frontière")
    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 6.5)
    ax.set_xlabel("b")
    ax.set_ylabel("c")
    ax.set_title("Nombre de racines de x² + bx + c")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.show()

## Partie C · Composition, réciproque et polynômes

Fiche §7 à §9. Composer en Python, prédire deux composées, une chasse à l'erreur, et le schéma de Horner.

### Ex R3.20 — Composer deux fonctions · `compose` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer une fonction qui fabrique une fonction (une fermeture), dans le bon ordre.  
**Prérequis :** fiche §7 ; workbookIA 0A §100.5.3 (fonctions comme valeurs) et §100.5.4 (fermetures) · **mymath :** `functions.py` · **Parcours :** C, E

Écris `compose(f, g)` dans `mon_travail/mymath/functions.py` : elle **renvoie une nouvelle fonction**, qui à `x` associe `f(g(x))`. Définis cette fonction à l'intérieur de `compose` (une fermeture, comme `then_add` dans la fiche §7), puis renvoie-la **sans l'appeler**.

La vérification fabrique $x \mapsto (x - 1)^2$ et $x \mapsto x^2 - 1$ avec `compose`, et les calcule en 4.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R3")  # reload your saved file

In [ ]:
with checked("R3.20"):
    square_after = mymath.functions.compose(lambda t: t**2, lambda t: t - 1)
    minus_after = mymath.functions.compose(lambda t: t - 1, lambda t: t**2)
    wbm.check("R3.20", [square_after(4), minus_after(4)], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r3_functions.py", "test_compose_")

### Ex R3.23 — Prédire f∘g et g∘f pour « valeur absolue » et « moins 3 » 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir le résultat d'une composée dans les deux ordres, puis le voir sur les courbes.  
**Prérequis :** fiche §7 · **Parcours :** C

On pose $f(x) = |x|$ (« valeur absolue ») et $g(x) = x - 3$ (« moins 3 »). **Avant d'exécuter quoi que ce soit**, prédis :
- a) $(f \circ g)(1)$ ;
- b) $(g \circ f)(1)$ ;
- c) $(f \circ g)(-2)$ ;
- d) $(g \circ f)(-2)$.

Écris tes quatre prédictions, vérifie-les, puis lance l'expérience : elle calcule les quatre valeurs et trace les deux composées. Dans ta copie : pour quels $x$ les deux composées sont-elles égales ? Pourquoi ?

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R3_23a = ...  # a) a number, written BEFORE running the experiment below
prediction_R3_23b = ...  # b) a number
prediction_R3_23c = ...  # c) a number
prediction_R3_23d = ...  # d) a number

In [ ]:
for letter, prediction in zip("abcd", [prediction_R3_23a, prediction_R3_23b, prediction_R3_23c,
                                       prediction_R3_23d]):
    wbm.check(f"R3.23{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R3_23a, prediction_R3_23b, prediction_R3_23c, prediction_R3_23d]):
    print("⏳ Écris d'abord tes quatre prédictions (des nombres), puis exécute à nouveau.")
else:
    f = abs
    g = lambda t: t - 3
    print("(f∘g)(1) =", f(g(1)), " ; (g∘f)(1) =", g(f(1)))
    print("(f∘g)(−2) =", f(g(-2)), " ; (g∘f)(−2) =", g(f(-2)))
    xs = np.linspace(-5, 7, 400)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(xs, np.abs(xs - 3), label="(f∘g)(x) = |x − 3|", linewidth=2.2)
    ax.plot(xs, np.abs(xs) - 3, "--", label="(g∘f)(x) = |x| − 3", linewidth=2.2)
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.axvline(0, color="gray", linewidth=0.6)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Deux ordres, deux fonctions")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

### Ex R3.25 — Chasse à l'erreur : composition, réciproque et racines 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer les erreurs classiques sur les composées, les réciproques et le second degré, et les corriger.  
**Prérequis :** fiche §5, §7 et §8, « Les pièges classiques » · **Parcours :** C, E

Un camarade a fait sept calculs. **Sans rien exécuter d'abord**, trouve ceux qui sont faux. On pose $f(x) = 5x - 1$ et $g(x) = x^2$.

| | Calcul |
|---|---|
| A | $(f \circ g)(x) = (5x - 1)^2$ |
| B | $f^{-1}(x) = \dfrac{1}{5x - 1}$ |
| C | les racines de $x^2 - 2x - 15$ sont $-5$ et $3$ |
| D | $3x^2 + 6x + 1 = 3(x + 1)^2 - 2$ |
| E | $x \mapsto 3 - x$ est décroissante sur $\mathbb{R}$ |
| F | le discriminant de $2x^2 - 3x - 4$ vaut $9 - 32 = -23$ |
| G | pour $4x^2 - 4x - 3$, $\Delta = 64$, donc les racines sont $4 \pm \frac{8}{8}$, soit 3 et 5 |

- a) Les lettres des calculs faux, dans un texte.
- b) Le bon $(f \circ g)(x)$ : une liste de coefficients.
- c) Les bonnes racines de $x^2 - 2x - 15$ : une liste.
- d) Le bon discriminant de $2x^2 - 3x - 4$.
- e) Les bonnes racines de $4x^2 - 4x - 3$ : une liste.

In [ ]:
answer_R3_25a = ...  # a) the letters of the wrong calculations, as text
answer_R3_25b = ...  # b) a list of coefficients
answer_R3_25c = ...  # c) a list
answer_R3_25d = ...  # d) a number
answer_R3_25e = ...  # e) a list

In [ ]:
for letter, answer in zip("abcde", [answer_R3_25a, answer_R3_25b, answer_R3_25c, answer_R3_25d, answer_R3_25e]):
    wbm.check(f"R3.25{letter}", answer)

### Ex R3.17 — Évaluer un polynôme par Horner · `polyval` 🔨 ★ ⏱️ 25 min
**Objectif :** programmer le schéma de Horner, qui évalue un polynôme sans calculer aucune puissance.  
**Prérequis :** fiche §9 ; workbookIA 0A §100.4.2 (boucle `for`) · **mymath :** `functions.py` · **Parcours :** C

> **Mode d'emploi `mymath`** (comme en R2). Ouvre ta copie `mon_travail/mymath/functions.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Les tests comparent tes résultats à NumPy et à SymPy : écris le calcul toi-même, sans les appeler.

Écris `polyval(coefficients, x)` : les coefficients vont du plus haut degré au terme constant (fiche §9). Pars de 0, puis, pour chaque coefficient, multiplie la valeur courante par `x` et ajoute le coefficient. Sans `**` : c'est tout l'intérêt du schéma. Avec des entiers, le résultat reste entier ; avec un tableau NumPy en `x`, les mêmes opérations marchent case par case.

La vérification calcule `polyval([1, -2, 0, 3], -2)` et `polyval([0.5, 1], 4)`.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R3")  # reload your saved file

In [ ]:
with checked("R3.17a"):
    wbm.check("R3.17a", mymath.functions.polyval([1, -2, 0, 3], -2), computed=True)
with checked("R3.17b"):
    wbm.check("R3.17b", mymath.functions.polyval([0.5, 1], 4), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r3_functions.py", "test_polyval_")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu images et antécédents, lis-tu un graphique et trouves-tu un ensemble de définition ?
2. Trouves-tu une droite par deux points, la forme canonique, les racines et le sommet d'un second degré, et un maximum sur un intervalle ?
3. Composes-tu dans le bon ordre, trouves-tu une réciproque, et sais-tu écrire `compose` et `polyval` en Python ?

**La suite** : R4, *Logique, ensembles, Σ et Π*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.